In [20]:
import pandas as pd

In [21]:
sf = pd.read_csv('stratification_frame_cleaned.csv')
sf.head

<bound method NDFrame.head of        eroll_no constituency_name  gender age_band area_type  R    G   SG  \
0           1.0         Mekliganj  Female    26-35       Rur  H  GEN  GUC   
1           1.0         Mekliganj    Male    36-45       Rur  H  GEN  GUC   
2           1.0         Mekliganj    Male    26-35       Rur  H  GEN  GUC   
3           1.0         Mekliganj  Female    36-45       Rur  H  GEN  GUC   
4           1.0         Mekliganj    Male    46-60       Rur  H  GEN  GUC   
...         ...               ...     ...      ...       ... ..  ...  ...   
66665       NaN         Tufanganj    Male    46-60       Rur  M  OBC   BM   
66666       NaN         Tufanganj    Male      60+       Rur  H  GEN  GUC   
66667       NaN         Tufanganj    Male      60+       Rur  H  OBC  MAT   
66668       NaN         Tufanganj    Male      60+       Rur  H   SC  RAJ   
66669       NaN         Tufanganj    Male      60+       Rur  H   ST  RAJ   

       n_individuals  stratum_share  
0      

In [22]:
sf.head()

,eroll_no,constituency_name,gender,age_band,area_type,R,G,SG,n_individuals,stratum_share
0,1.0,Mekliganj,Female,26-35,Rur,H,GEN,GUC,283,0.080880
1,1.0,Mekliganj,Male,36-45,Rur,H,GEN,GUC,274,0.078308
2,1.0,Mekliganj,Male,26-35,Rur,H,GEN,GUC,272,0.077736
3,1.0,Mekliganj,Female,36-45,Rur,H,GEN,GUC,244,0.069734
4,1.0,Mekliganj,Male,46-60,Rur,H,GEN,GUC,213,0.060875


In [23]:
print('Sum of n_individuals:', sf['n_individuals'].sum())

Sum of n_individuals: 957304


In [24]:
merged = pd.read_csv('merged_sample_v1.csv')
merged['constituency_name'].nunique()

265

In [25]:
df = sf

In [26]:
sf['constituency_name'].nunique()

294

In [27]:
sf_constituencies = set(sf['constituency_name'].dropna().unique())
merged_constituencies = set(merged['constituency_name'].dropna().unique())
missing = sorted(sf_constituencies - merged_constituencies)
print(f"Total missing: {len(missing)}")
for c in missing:
    print(c)

Total missing: 29
Ausgram
Behala Paschim
Behala Purba
Beleghata
Bhatar
Bidhannagar
Burdwan Uttar
Darjeeling
Durgapur Paschim
Durgapur Purba
Entally
Galsi
Jorasanko
Kalimpong
Kasba
Kashipur Belgachhia
Katwa
Ketugram
Kharagpur Sadar
Kurseong
Mangalkot
Maniktola
Metiaburuz
Pandaveswar
Purbasthali Dakshin
Purbasthali Uttar
Rajarhat New Town
Rashbehari
Shyampukur


In [28]:
import pandas as pd
import numpy as np

TOTAL_SAMPLE = 5_000
SEED = 42

# Identify missing constituencies
sf_constituencies = set(sf['constituency_name'].dropna().unique())
merged_constituencies = set(merged['constituency_name'].dropna().unique())
missing = sf_constituencies - merged_constituencies

# Filter sf to only missing constituencies
sf_missing = sf[sf['constituency_name'].isin(missing)].copy()
sf_valid_missing = sf_missing[sf_missing['n_individuals'] > 0].copy()

print(f"Missing constituencies: {len(missing)}")
print(f"Valid strata rows:      {len(sf_valid_missing):,}")
print(f"Total individuals:      {sf_valid_missing['n_individuals'].sum():,}")

# Step 1: Guarantee 1 row per missing constituency (largest stratum)
guaranteed = (
    sf_valid_missing.sort_values('n_individuals', ascending=False)
    .groupby('constituency_name', sort=False)
    .head(1)
    .reset_index(drop=True)
)

# Step 2: Weighted sample for remaining budget
remaining = TOTAL_SAMPLE - len(guaranteed)

weights = sf_valid_missing['n_individuals'] / sf_valid_missing['n_individuals'].sum()
extra = sf_valid_missing.sample(n=remaining, replace=True, weights=weights, random_state=SEED).reset_index(drop=True)

# Step 3: Combine
sample_missing = pd.concat([guaranteed, extra], ignore_index=True)

print(f"\nTotal rows:             {len(sample_missing):,}")
print(f"Constituencies covered: {sample_missing['constituency_name'].nunique()} / {len(missing)}")
print(f"Duplicate rows:         {sample_missing.duplicated().sum():,}")

sample_missing.to_csv('sample_10k_missing_constituencies.csv', index=False)
print("Saved sample_10k_missing_constituencies.csv")

Missing constituencies: 29
Valid strata rows:      6,467
Total individuals:      65,189

Total rows:             5,000
Constituencies covered: 29 / 29
Duplicate rows:         3,121
Saved sample_10k_missing_constituencies.csv


In [29]:

df_unique = (
    sample_missing
    .groupby(sample_missing.columns.tolist(), as_index=False)
    .size()
    .rename(columns={'size': 'dup_count'})
)

# Save to CSV
df_unique.to_csv('sample_missing_dedup_with_counts.csv', index=False)

In [30]:
df_unique.head()

,eroll_no,constituency_name,gender,age_band,area_type,R,G,SG,n_individuals,stratum_share,dup_count
0,266.0,Burdwan Uttar,Female,18-25,Rur,H,GEN,GUC,69,0.018548,5
1,266.0,Burdwan Uttar,Female,18-25,Rur,H,OBC,MAH,10,0.002688,1
2,266.0,Burdwan Uttar,Female,18-25,Rur,H,SC,RAJ,2,0.000538,1
3,266.0,Burdwan Uttar,Female,18-25,Rur,M,OBC,BM,15,0.004032,1
4,266.0,Burdwan Uttar,Female,18-25,Urb,H,OBC,MAH,1,0.000269,1


In [31]:
df_unique.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017 entries, 0 to 1016
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   eroll_no           1017 non-null   float64
 1   constituency_name  1017 non-null   object 
 2   gender             1017 non-null   object 
 3   age_band           1017 non-null   object 
 4   area_type          1017 non-null   object 
 5   R                  1017 non-null   object 
 6   G                  1017 non-null   object 
 7   SG                 1017 non-null   object 
 8   n_individuals      1017 non-null   int64  
 9   stratum_share      1017 non-null   float64
 10  dup_count          1017 non-null   int64  
dtypes: float64(2), int64(2), object(7)
memory usage: 87.5+ KB


In [32]:
import numpy as np
for i, chunk in enumerate(np.array_split(df_unique, 4), start=9):
    chunk.to_csv(f'sample_{i}.csv', index=False)
    print(f"sample_{i}.csv: {len(chunk):,} rows")


sample_9.csv: 255 rows
sample_10.csv: 254 rows
sample_11.csv: 254 rows
sample_12.csv: 254 rows


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [9]:
import pandas as pd
import numpy as np

TOTAL_SAMPLE = 50_000
SEED = 42

# Filter valid strata
sf_valid = sf[sf['n_individuals'] > 0].copy()

# Step 1: Guarantee 1 row per constituency (pick the largest stratum per constituency)
guaranteed = (
    sf_valid.sort_values('n_individuals', ascending=False)
    .groupby('constituency_name', sort=False)
    .head(1)
    .reset_index(drop=True)
)  # 294 rows

# Step 2: Weighted sample for remaining budget
remaining = TOTAL_SAMPLE - len(guaranteed)  # 49,706 rows

weights = sf_valid['n_individuals'] / sf_valid['n_individuals'].sum()
extra = sf_valid.sample(n=remaining, replace=True, weights=weights, random_state=SEED).reset_index(drop=True)

# Step 3: Combine
sample_df = pd.concat([guaranteed, extra], ignore_index=True)

print(f"Total rows:             {len(sample_df):,}")
print(f"Constituencies covered: {sample_df['constituency_name'].nunique()} / 294")
print(f"Duplicate rows:         {sample_df.duplicated().sum():,}")

sample_df.to_csv('sample_50k_sf.csv', index=False)
print("Saved sample_50k_sf.csv")


Total rows:             50,000
Constituencies covered: 294 / 294
Duplicate rows:         29,809
Saved sample_50k_sf.csv


In [10]:
sample_allocation = sample_df

In [11]:
sample_allocation.to_csv('sample_allocation.csv', index=False)

In [1]:
import pandas as pd
sample_allocation = pd.read_csv('sample_allocation.csv')
sample_allocation.head()

,eroll_no,constituency_name,gender,age_band,area_type,R,G,SG,n_individuals,stratum_share
0,150.0,Jadavpur,Female,46-60,Urb,H,GEN,GUC,549,0.112224
1,110.0,Dum Dum Uttar,Female,46-60,Urb,H,GEN,GUC,460,0.109290
2,114.0,Dum Dum,Female,46-60,Urb,H,GEN,GUC,417,0.101856
3,152.0,Tollyganj,Female,46-60,Urb,H,GEN,GUC,399,0.098689
4,106.0,Jagatdal,Male,46-60,Urb,H,GEN,GUC,390,0.091228


In [4]:

df_unique = (
    sample_allocation
    .groupby(sample_allocation.columns.tolist(), as_index=False)
    .size()
    .rename(columns={'size': 'dup_count'})
)

# Save to CSV
df_unique.to_csv('sample_allocation_dedup_with_counts.csv', index=False)

In [2]:
sample_allocation.duplicated().sum()

np.int64(29809)

In [33]:
df_unique.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017 entries, 0 to 1016
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   eroll_no           1017 non-null   float64
 1   constituency_name  1017 non-null   object 
 2   gender             1017 non-null   object 
 3   age_band           1017 non-null   object 
 4   area_type          1017 non-null   object 
 5   R                  1017 non-null   object 
 6   G                  1017 non-null   object 
 7   SG                 1017 non-null   object 
 8   n_individuals      1017 non-null   int64  
 9   stratum_share      1017 non-null   float64
 10  dup_count          1017 non-null   int64  
dtypes: float64(2), int64(2), object(7)
memory usage: 87.5+ KB


In [34]:
df_unique.to_csv('sample_allocation_dedup_with_counts.csv', index=False)

In [7]:
import numpy as np
for i, chunk in enumerate(np.array_split(df_unique, 4), start=1):
    chunk.to_csv(f'sample_{i}.csv', index=False)
    print(f"sample_{i}.csv: {len(chunk):,} rows")


sample_1.csv: 4,810 rows
sample_2.csv: 4,810 rows
sample_3.csv: 4,810 rows
sample_4.csv: 4,809 rows


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
